# 📖 Notebook 3: Backup Strategies

## Why This Matters

Replication protects you from hardware failure, but NOT from:
- **Accidental DELETE** — someone runs `DELETE FROM orders` without a WHERE clause
- **Ransomware** — malicious encryption of your data (replicated to standby!)
- **Logical corruption** — a bug writes bad data to the database
- **Compliance** — regulations require you to keep historical backups

**Replication is NOT a backup.** If you delete data on the primary,
the delete is replicated to the standby. You need actual backups.

## Learning Objectives

- Understand full, incremental, and differential backup types
- Perform a logical backup with `pg_dump`
- Perform a physical backup with `pg_basebackup`
- Test backup restoration (the most neglected practice!)
- Understand point-in-time recovery (PITR) concepts

## 🛠️ Setup

```bash
cd enterprise-patterns/bcdr
docker-compose down -v && docker-compose up -d
```

This resets the environment to a clean state.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).

In [ ]:
import psycopg2
import subprocess
import time
import os
from tabulate import tabulate

DB_PRIMARY = {
    "host": "localhost", "port": 5432,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def docker_exec(container, cmd):
    result = subprocess.run(
        ["docker", "exec", container] + cmd,
        capture_output=True, text=True, timeout=60
    )
    return result.stdout.strip(), result.stderr.strip()

# Test connection
conn = get_primary_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM orders")
print(f"✅ Connected. Orders in database: {cur.fetchone()[0]}")
conn.close()

## 📚 Backup Types Explained

### Full Backup
- Copies **everything** in the database
- Slowest to create, largest file size
- Fastest to restore (just load one file)
- **Example**: `pg_dump` or `pg_basebackup`

### Incremental Backup
- Copies only data that changed **since the last backup of any type**
- Fastest to create, smallest file size
- Slowest to restore (need full + every incremental in order)
- **Example**: WAL archiving between `pg_basebackup` runs

### Differential Backup
- Copies only data that changed **since the last full backup**
- Medium speed to create, medium file size
- Medium restore speed (need full + latest differential)

```
         Day 1       Day 2       Day 3       Day 4       Day 5
Full:   [AAABBB]
Incr:               [CC]        [DD]        [EE]        [FF]
Diff:               [CC]        [CCDD]      [CCDDEE]    [CCDDEEFF]

To restore Day 5:
  Full only:  Not possible (Day 1 data only)
  Incremental: Full + CC + DD + EE + FF (5 files)
  Differential: Full + CCDDEEFF (2 files)
```

In [ ]:
# =============================================================================
# Demo: Logical Backup with pg_dump
# =============================================================================
# pg_dump creates a SQL script that recreates your database.
# It is a LOGICAL backup — human-readable SQL statements.

print("=" * 65)
print("LOGICAL BACKUP WITH pg_dump")
print("=" * 65)

# Create a full logical backup
start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_dump",
    "-U", "demo",
    "-d", "bcdr_demo",
    "--format=custom",       # compressed binary format
    "--file=/tmp/backup.dump"
])
backup_time = time.time() - start_time

if err and 'error' in err.lower():
    print(f"❌ Backup failed: {err}")
else:
    print(f"  ✅ Backup completed in {backup_time:.2f} seconds")

    # Check backup size
    out, _ = docker_exec('bcdr-pg-primary', [
        "ls", "-lh", "/tmp/backup.dump"
    ])
    print(f"  📦 Backup file: {out}")

    # Verify backup contents (list what is inside)
    out, _ = docker_exec('bcdr-pg-primary', [
        "pg_restore", "--list", "/tmp/backup.dump"
    ])
    lines = out.strip().split('\n')
    print(f"  📋 Backup contains {len(lines)} objects")
    print("\n  First 10 objects in backup:")
    for line in lines[:10]:
        print(f"    {line}")

## 📚 Physical Backup with pg_basebackup

`pg_basebackup` creates a **physical copy** of the entire database cluster.
This is the same tool we used to set up our standby server!

### Logical vs Physical Backup

| Feature | Logical (pg_dump) | Physical (pg_basebackup) |
|---------|------------------|-------------------------|
| What it copies | Table data + schema | Entire data directory (binary) |
| Speed | Slower (reads every row) | Faster (copies files) |
| Selective restore | Yes (individual tables) | No (all or nothing) |
| Cross-version | Yes (can restore to newer PG) | No (same major version only) |
| Point-in-time recovery | No | Yes (with WAL archiving) |
| Best for | Small databases, migrations | Large databases, PITR |

In [ ]:
# =============================================================================
# Demo: Physical Backup with pg_basebackup
# =============================================================================

print("=" * 65)
print("PHYSICAL BACKUP WITH pg_basebackup")
print("=" * 65)

start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_basebackup",
    "-U", "demo",
    "-D", "/tmp/physical_backup",
    "-Ft",    # tar format
    "-z",     # gzip compression
    "-Xs",    # stream WAL during backup
    "-P"      # show progress
])
backup_time = time.time() - start_time

print(f"  Time: {backup_time:.2f} seconds")
if err:
    # pg_basebackup outputs progress to stderr
    progress_lines = [l for l in err.split('\n') if '%' in l]
    if progress_lines:
        print(f"  Progress: {progress_lines[-1].strip()}")

# Check backup files
out, _ = docker_exec('bcdr-pg-primary', [
    "ls", "-lh", "/tmp/physical_backup/"
])
print(f"\n  Backup files:")
for line in out.split('\n'):
    if line.strip():
        print(f"    {line.strip()}")

print("\n💡 base.tar.gz = all database files, pg_wal.tar.gz = WAL logs")
print("   Together these can restore the database to this exact point in time.")

## 📚 Backup Verification — The Most Neglected Practice

> **An untested backup is not a backup — it is a hope.**

Many organizations discover their backups are corrupted or incomplete
only when they need to restore them during a real disaster.

### Backup Verification Checklist

1. **Can you restore it?** — Actually restore to a test database
2. **Is the data complete?** — Compare row counts, checksums
3. **How long does restore take?** — This is your actual RTO for backup-based recovery
4. **Can someone else do it?** — Document the procedure, test with different team members
5. **Is the backup accessible?** — Can you reach it during a disaster?

In [ ]:
# =============================================================================
# Demo: Restore and Verify a Backup
# =============================================================================
# We will restore our pg_dump backup to a different database and verify it.

print("=" * 65)
print("BACKUP VERIFICATION: Restore + Verify")
print("=" * 65)

# Step 1: Create a test database for restoration
print("\nStep 1: Creating test database for restore...")
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "DROP DATABASE IF EXISTS bcdr_restore_test"
])
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "CREATE DATABASE bcdr_restore_test"
])
print("  ✅ Test database created")

# Step 2: Restore the backup
print("\nStep 2: Restoring backup...")
start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_restore",
    "-U", "demo",
    "-d", "bcdr_restore_test",
    "--no-owner",
    "/tmp/backup.dump"
])
restore_time = time.time() - start_time
print(f"  ✅ Restore completed in {restore_time:.2f} seconds")

# Step 3: Verify data integrity
print("\nStep 3: Verifying data integrity...")

conn_orig = get_primary_connection()
cur_orig = conn_orig.cursor()

conn_rest = psycopg2.connect(
    host="localhost", port=5432,
    database="bcdr_restore_test", user="demo", password="demo"
)
cur_rest = conn_rest.cursor()

tables = ['customers', 'orders', 'order_items', 'payments', 'audit_log']
table_data = []
all_match = True

for tbl in tables:
    cur_orig.execute(f"SELECT COUNT(*) FROM {tbl}")
    orig_count = cur_orig.fetchone()[0]
    cur_rest.execute(f"SELECT COUNT(*) FROM {tbl}")
    rest_count = cur_rest.fetchone()[0]
    match = "✅" if orig_count == rest_count else "❌"
    if orig_count != rest_count:
        all_match = False
    table_data.append([tbl, orig_count, rest_count, match])

print(tabulate(table_data,
    headers=["Table", "Original", "Restored", "Match"],
    tablefmt="grid"))

if all_match:
    print("\n🎉 All row counts match! Backup is verified.")
else:
    print("\n⚠️  Row count mismatch detected!")

print(f"\n📊 Restore time: {restore_time:.2f}s — this is your backup-based RTO")

conn_orig.close()
conn_rest.close()

# Cleanup
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "DROP DATABASE IF EXISTS bcdr_restore_test"
])

## 📚 Point-in-Time Recovery (PITR)

PITR lets you restore your database to **any specific moment in time**.

```
                  pg_basebackup    accidental    restore
  ─────────────────[BACKUP]─────────[DELETE]──────[HERE]──────
                     ▲                              ▲
                     │     WAL logs fill the gap    │
                     │◄────────────────────────────►│
```

### How It Works

1. Start with a `pg_basebackup` (your base snapshot)
2. PostgreSQL continuously archives WAL files (the change log)
3. To recover: restore the base backup, then replay WAL files up to your target time
4. This gets you to any point between the backup and the disaster

### Requirements
- `archive_mode = on` in PostgreSQL config (we have this!)
- WAL archive storage that survives the disaster
- A known good timestamp to recover to

In [ ]:
# =============================================================================
# Demo: Check WAL Archiving Status
# =============================================================================

conn = get_primary_connection()
cur = conn.cursor()

# Check archive settings
cur.execute(
    "SELECT name, setting FROM pg_settings "
    "WHERE name IN ('archive_mode', 'archive_command', 'wal_level') "
    "ORDER BY name"
)
settings = cur.fetchall()

print("=" * 65)
print("WAL ARCHIVE CONFIGURATION")
print("=" * 65)
for name, value in settings:
    print(f"  {name}: {value}")

# Check archived files
cur.execute(
    "SELECT archived_count, failed_count, "
    "last_archived_wal, last_archived_time "
    "FROM pg_stat_archiver"
)
row = cur.fetchone()
print(f"\n  Archived WAL files: {row[0]}")
print(f"  Failed archives:    {row[1]}")
print(f"  Last archived WAL:  {row[2]}")
print(f"  Last archive time:  {row[3]}")

conn.close()

print("\n💡 WAL archiving + pg_basebackup = point-in-time recovery capability.")
print("   This is how enterprises achieve RPO of minutes or even seconds.")

## 📝 Summary

### What You Learned

1. **Replication is NOT backup** — Deletes and corruption replicate too!
2. **Full/Incremental/Differential** — Trade-offs between backup speed, size, and restore speed.
3. **pg_dump** — Logical backup (SQL). Good for small DBs, selective restore, cross-version.
4. **pg_basebackup** — Physical backup (binary). Good for large DBs and PITR.
5. **Backup verification** — Always restore and verify. Measure your actual restore time.
6. **PITR** — Base backup + WAL archive = restore to any point in time.

### Key Takeaway

> **An untested backup is not a backup. Schedule regular restore tests.**

### Next Notebook

In **Notebook 4**, we run a full disaster recovery drill — simulating
a primary failure and measuring our actual RTO.